## Обучение модели

В этом ноутбуке обучим несколько моделей для предсказания оценки по математике и выберем лучшую.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# модели
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from catboost import CatBoostRegressor

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('data/stud.csv')
df.head()

### Подготовка данных

In [ ]:
# целевая переменная и признаки
X = df.drop(columns=['math score'])
y = df['math score']

print(f'Размер X: {X.shape}')
print(f'Размер y: {y.shape}')

In [ ]:
# разбиваем на train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
# определяем числовые и категориальные колонки
num_cols = ['reading score', 'writing score']
cat_cols = ['gender', 'race/ethnicity', 'parental level of education', 'lunch', 'test preparation course']

print(f'Числовые: {num_cols}')
print(f'Категориальные: {cat_cols}')

In [ ]:
# создаём препроцессор
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
])

# трансформируем данные
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)

print(f'После препроцессинга: {X_train_scaled.shape[1]} признаков')

### Функция для оценки моделей

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    """Обучает модель и возвращает метрики"""
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    metrics = {
        'train_r2': r2_score(y_train, y_train_pred),
        'test_r2': r2_score(y_test, y_test_pred),
        'train_rmse': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'test_rmse': np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'test_mae': mean_absolute_error(y_test, y_test_pred)
    }
    
    return metrics, y_test_pred

### Обучение моделей

In [ ]:
# словарь с моделями для тестирования
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.1),
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    'AdaBoost': AdaBoostRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    'CatBoost': CatBoostRegressor(iterations=100, learning_rate=0.1, depth=6, verbose=False, random_state=42)
}

In [ ]:
# обучаем все модели и собираем результаты
results = {}
predictions = {}

for name, model in models.items():
    metrics, preds = evaluate_model(model, X_train_scaled, X_test_scaled, y_train, y_test)
    results[name] = metrics
    predictions[name] = preds
    print(f'{name}: R2={metrics["test_r2"]:.4f}, RMSE={metrics["test_rmse"]:.2f}')

In [ ]:
# сводная таблица
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('test_r2', ascending=False)
results_df.round(4)

### Визуализация результатов

In [ ]:
# сравнение R2 score
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(results_df))
width = 0.35

ax.bar(x - width/2, results_df['train_r2'], width, label='Train R2')
ax.bar(x + width/2, results_df['test_r2'], width, label='Test R2')

ax.set_ylabel('R2 Score')
ax.set_title('Сравнение моделей по R2')
ax.set_xticks(x)
ax.set_xticklabels(results_df.index, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

### Анализ лучшей модели

In [ ]:
# лучшая модель по test R2
best_model_name = results_df['test_r2'].idxmax()
best_preds = predictions[best_model_name]

print(f'Лучшая модель: {best_model_name}')
print(f'Test R2: {results_df.loc[best_model_name, "test_r2"]:.4f}')
print(f'Test RMSE: {results_df.loc[best_model_name, "test_rmse"]:.2f}')

In [ ]:
# график предсказаний vs реальных значений
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# scatter plot
axes[0].scatter(y_test, best_preds, alpha=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Реальные значения')
axes[0].set_ylabel('Предсказания')
axes[0].set_title(f'{best_model_name}: предсказания vs реальность')

# распределение ошибок
errors = y_test - best_preds
axes[1].hist(errors, bins=20, edgecolor='black')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Ошибка (реальное - предсказание)')
axes[1].set_ylabel('Частота')
axes[1].set_title('Распределение ошибок')

plt.tight_layout()
plt.show()

In [ ]:
# примеры предсказаний
comparison = pd.DataFrame({
    'actual': y_test.values[:15],
    'predicted': best_preds[:15].round(1),
    'error': (y_test.values[:15] - best_preds[:15]).round(1)
})
comparison

### Выводы

Результаты обучения:

1. Линейная регрессия показала лучший результат с R2 около 0.88
2. Простые линейные модели (Linear, Ridge, Lasso) работают на уровне или лучше ансамблей
3. Это объясняется высокой корреляцией между оценками (reading и writing хорошо предсказывают math)
4. Decision Tree показывает переобучение (train R2 высокий, test R2 низкий)
5. RMSE около 5-6 баллов — приемлемая точность для 100-балльной шкалы

Для продакшена выбираем Linear Regression как самую простую и эффективную модель.